In [1]:
import json
from pathlib import Path
from datasets import load_dataset


In [9]:

# ───────────────── Configuration - TEST ─────────────────────────────────────────────
ALIGN_PATH   = Path("../alignment_checks_test_gpt-4o.json")
BASE_PATH    = Path("../../responses/base/generated_responses_test.json")
GRPO_PATH    = Path("../../responses/grpo/generated_responses_test.json")
OUT_PATH     = Path("human_label_test_split.json")
DATASET_NAME = "cambridgeltl/vsr_zeroshot"
# ──────────────────────────────────────────────────────────────────────────────

def extract_model_reasoning(full_resp: str) -> str:
    """
    Split off the chain‐of‐thought reasoning (everything before '\nAnswer:').
    If no explicit marker, take all but the last line.
    """
    if "\nAnswer:" in full_resp:
        reasoning, _ = full_resp.rsplit("\nAnswer:", 1)
    else:
        lines = full_resp.strip().splitlines()
        reasoning = "\n".join(lines[:-1])
    return reasoning.strip()

def main():
    # 1) Load the judge’s test‐split output
    alignment = json.loads(ALIGN_PATH.read_text())

    # 2) Load model responses
    base_out = json.loads(BASE_PATH.read_text())
    grpo_out = json.loads(GRPO_PATH.read_text())

    # 3) Load the first 350 of the test split to get image URLs
    vsr = load_dataset(DATASET_NAME, split="test")
    vsr = vsr.select(range(350))

    detailed = []
    for entry in alignment:
        idx   = entry["index"]
        model = entry["model"]

        # pick the corresponding response
        if model == "base":
            resp_obj = base_out[idx]
        elif model == "grpo":
            resp_obj = grpo_out[idx]
        else:
            raise ValueError(f"Unknown model '{model}'")

        full_pred = resp_obj.get("predicted_response", "").strip()

        merged = {
            **entry,  # keeps: index, model, question, true_label, final_ans, aligns, why
            "image_url":          vsr[idx]["image_link"],
            "model_reasoning":    extract_model_reasoning(full_pred),
            # "predicted_response": full_pred,
            # optional: include the true_response too
            # "true_response":      resp_obj.get("true_response", "").strip(),
            "human_aligns":       "",   # ← fill in with "True"/"False"
            "human_comment":      ""    # ← optional free‐text note
        }
        detailed.append(merged)

    # 4) Write out
    OUT_PATH.write_text(json.dumps(detailed, indent=2))
    print(f"Wrote {len(detailed)} entries → {OUT_PATH}")

if __name__ == "__main__":
    main()

Wrote 700 entries → human_label_test_split.json


In [ ]:
# If running in Jupyter, install ipywidgets:
# !pip install ipywidgets

import json
from pathlib import Path
from IPython.display import display, HTML, clear_output

import ipywidgets as widgets

# Paths
INPUT_PATH = Path("human_label_test_split.json")
OUTPUT_PATH = Path("human_label_test_split_filled.json")

# Load or initialize data
if OUTPUT_PATH.exists():
    with open(OUTPUT_PATH, 'r') as f:
        data = json.load(f)
else:
    with open(INPUT_PATH, 'r') as f:
        data = json.load(f)

n = len(data)
current = 0

# Widgets
align_widget = widgets.RadioButtons(
    options=["True", "False"],
    description="Human aligns:",
    style={'description_width': 'initial'}
)
comment_widget = widgets.Textarea(
    placeholder='Enter any comments here...',
    description='Human comment:',
    layout=widgets.Layout(width='600px', height='100px'),
    style={'description_width': 'initial'}
)
prev_button = widgets.Button(description="<< Previous", button_style='')
next_button = widgets.Button(description="Next >>", button_style='primary')

def save_progress():
    """Save current state to OUTPUT_PATH."""
    with open(OUTPUT_PATH, 'w') as f:
        json.dump(data, f, indent=2)

def show_entry(i):
    """Display entry i and populate widgets."""
    clear_output(wait=True)
    entry = data[i]
    # Populate widgets with existing values
    align_widget.value = entry.get("human_aligns") or None
    comment_widget.value = entry.get("human_comment", "")
    
    # Build HTML
    left_html = "<ul>"
    for key, label in [
        ("question", "Question"),
        ("true_label", "True Label"),
        ("final_ans", "Model Answer"),
        ("aligns", "Judge Aligns"),
        ("why", "Judge Reason"),
        ("model_reasoning", "Model Reasoning"),
    ]:
        if key in entry:
            left_html += f"<li><b>{label}:</b> {entry[key]}</li>"
    left_html += "</ul>"
    html = f"""
    <div style="display: flex; gap: 20px; align-items: flex-start;">
      <div style="flex: 1; max-width: 50%; overflow-wrap: break-word;">
        <h3>Entry {i+1}/{n} (Index: {entry['index']}, Model: {entry['model']})</h3>
        {left_html}
      </div>
      <div style="flex: 1; text-align: center;">
        <img src="{entry['image_url']}" style="max-width: 100%; height: auto; border:1px solid #ccc;"/>
      </div>
    </div>
    """
    display(HTML(html))
    # Button container with spacing
    button_box = widgets.HBox([prev_button, next_button], layout=widgets.Layout(justify_content='space-between'))
    display(align_widget, comment_widget, button_box)

def on_prev_clicked(b):
    global current
    # Save current inputs
    data[current]['human_aligns'] = align_widget.value
    data[current]['human_comment'] = comment_widget.value
    save_progress()
    # Move back
    if current > 0:
        current -= 1
    show_entry(current)

def on_next_clicked(b):
    global current
    # Save current inputs
    data[current]['human_aligns'] = align_widget.value
    data[current]['human_comment'] = comment_widget.value
    save_progress()
    # Move forward
    if current < n - 1:
        current += 1
        show_entry(current)
    else:
        clear_output()
        display(HTML(f"<h2>All entries labeled up to entry {n}.</h2>"
                     f"<p>Progress saved to <code>{OUTPUT_PATH}</code></p>"))

# Bind events
prev_button.on_click(on_prev_clicked)
next_button.on_click(on_next_clicked)

# Initial display
show_entry(current)


RadioButtons(description='Human aligns:', options=('True', 'False'), style=DescriptionStyle(description_width=…

Textarea(value='', description='Human comment:', layout=Layout(height='100px', width='600px'), placeholder='En…